# Engenharia de Features para Regras de Anomalia

> Objetivo: preparar variáveis derivadas e flags para melhorar a detecção de anomalias no Censo Agro.

## Fluxo adotado
1. Carregar e tratar a base de estabelecimentos.
2. Construir features por tema (estabelecimentos, lavoura temporária e pecuária).
3. Criar flags de apoio para regras críticas.
4. Garantir consistência final sem valores nulos.

In [ ]:
import pandas as pd
import sqlite3

COLUNHAS_CHAVES = ['V010100', 'NUM_QUADRA', 'NUM_FACE', 'V010800']
VALOR_CATEGORICO_VAZIO = '<EMPTY>'
VALOR_CATEGORICO_NAO = '1'

## Funções utilitárias padronizadas

> Apenas funções realmente reutilizadas no notebook. As demais transformações usam métodos diretos do pandas.

In [ ]:
def preencher_nulos_dataframe(df, valor_texto=VALOR_CATEGORICO_VAZIO):
    """Preenche nulos com regra única para evitar valores ausentes após merges e concatenações."""
    df = df.copy()

    cols_numericas = df.select_dtypes(include=['number']).columns
    cols_categoricas = df.select_dtypes(include=['category']).columns
    cols_textuais = df.select_dtypes(include=['object', 'string']).columns

    if len(cols_numericas) > 0:
        df[cols_numericas] = df[cols_numericas].fillna(0)

    for col in cols_categoricas:
        if valor_texto not in df[col].cat.categories:
            df[col] = df[col].cat.add_categories([valor_texto])
        df[col] = df[col].fillna(valor_texto)

    cols_textuais_sem_categoria = cols_textuais.difference(cols_categoricas)
    if len(cols_textuais_sem_categoria) > 0:
        df[cols_textuais_sem_categoria] = (
            df[cols_textuais_sem_categoria]
            .astype('string')
            .fillna(valor_texto)
        )

    return df

# Estabelecimentos

In [ ]:
query = """
SELECT *
FROM estabel
"""

estabel_var_to_ignore = [
    "COD_SEQ_ESPECIE", "V010003", "V010004", "V011300", "V011401", "V011402",
    "V01150101", "V01150102", "V01160100", "V01160101", "V01180100", "V01180101",
    "V01180102", "V01180103", "V01170100", "V01170200", "V01170300", "V05010101",
    "V02020401", "V02190000", "V02190001", "V02040000", "V02050000", "V02050101",
    "V02050102", "V02060000", "V02060100", "V03030000", "V03040000", "V03050000", 
    "V03060000", "V03070000", "V03080000", "V04020000", "V04030000", "V04040000",
    "V04050000", "V04060000", "V04070000", "V04080000", "V04090000", "V04100000",
    "V04110000", "V04120000", "V05270104", "V05340101", "V05340102", "V05340103",
    "V05340104", "V05340105", "V05340106", "V05340107", "V05340108", "V05340109",
    "V05340110", "V05340111", "V48010101", "V48020000", "V48030000", 
    "V48040000", "V48050100", "V48050200", "V48050300", "V48060000", "V48070100", 
    "V48070200", "V48070300", "V48070101", "V48070102", "V48070201", "V48070202",
    "V48080100", "V48080200", "V48090000", "V50000000", "V51010000", "V51010101",
    "V01150000", "V51010100",
]

estabel_var_nulas = [
    "V010300", "V010010", "V010020", "V010400", "V010500", "V010600", "V010700", 
    "V010701", "V010702", "V010703", "V010704", "V010705", "V011000", "V011101", 
    "V011102", "V011103", "V011104", "V011105", "V011106", "V011107", "V011108", 
    "V011109", "V011110", "V011111", "V011112", "V011113", "V011114", "V011115", 
    "V011200", "V011201", "V011202", "V011203", "V011204", "V011205", "V011206", 
    "V011207", "V011208", "V011209", "V011210", "V011211", "V011212", "V011213", 
    "V48070501","V48070502", "V48070601", "V48070602", "V48070301", "V48070302", 
    "V48070401", "V48070402", "VW46014800", "VW46014900", "VW52133400", "VW52133401",  
    "V011214", "VW02170000", "VW04190000", "VW04200000", "VW04210000", "VW04220000", 
    "VW04240000", "VW04250000", "VW04260000", "VW04280000", "VW05340113", "V09010000", 
    "VW08012305", "VW10000002", "V37010100", "VW45013700", "VW45013800", "VW46012114", 
    "VW46012215", "VW46012316", "VW46012400", "VW46012519", "VW46012620", "VW46012721", 
    "VW46012800", "VW46012922", "VW46013000", "VW46013001", "VW46013127", "VW46013229", 
    "VW46013331", "VW46013427", "VW46013530", "VW46013600", "VW46013700", "VW46013838", 
    "VW46013939", "VW46014037", "VW46014135", "VW46014136", "VW46014241", "VW46014300", 
    "VW46014400", "VW46014542", "VW46011800", "VW46014500", "VW46014600", "VW46014700", 
    "VW52133402", "VW52133403", "VW53010000", "VW53020000", "VW08010403", "VW12030100",
    "VW02200000", "VW52140000", "VW52150000", "VW52160000", "VW52141516", "VW52190000",
    "VW52320000", "VW52210000", "VW52192021", "VW52220000", "VW52232424", "VW52232425",
    "VW52222300", "VW52280000", "VW52290000", "VW52310000", "VW52270000", "VW52013530",
    "VW52272800", "VW52143200", "VW52380000", "VW52390000", "VW52370000", "VW52340000",
    "VW52350000", "VW52410000", "VW52354100", "VW52144100", "VW52360100", "VW52014800",
    "VW52014900", "VW52143603", "V02240001"
]

# Variáveis que armazena os valores recebidos de recursos externos, que foram mapeadas incorretamente como object
variaveis_recursos_externos = [
    'V46010900', 'V46011000', 'V46011100', 
    'V46011200', 'V46011300'
]

with sqlite3.connect('../data/amstr_geral_criticas.db') as conn:
    df_estabel = pd.read_sql_query(query, conn).drop(columns=['id'])

df_estabel = df_estabel.loc[:, ~df_estabel.columns.isin(estabel_var_to_ignore + estabel_var_nulas)]
df_estabel = df_estabel.loc[:, ~df_estabel.columns.duplicated()]

df_estabel[variaveis_recursos_externos] = df_estabel[variaveis_recursos_externos].apply(pd.to_numeric)

df_estabel

Definindo as regras de imputação dos valores nulos para cada coluna restante na base de dados

In [ ]:
estabel_vars_numericas_zero = [
    "V010005", "VW01170300", "VW04020000", "VW04030000", "VW04040000",
    "VW04050000", "VW04060000", "VW04070000", "VW04080000", "VW04090000",
    "VW04100000", "VW04110000", "VW04120000", "VW04160000", "VW04170000",
    "VW04180000", "VW03030000", "VW03040000", "VW03050000", "VW03060000",
    "VW03070000", "VW03080000", "VW05270104", "VW05340101", "VW05340102",
    "VW05340103", "VW05340104", "VW05340105", "VW05340106", "VW05340107",
    "VW05340108", "VW05340109", "VW05340110", "VW05340111", "VW05340112",
    "V06030101", "V06030102", "V06040300", "V06040400", "V06040500",
    "V06040600", "V06040700", "V06040800", "V07020101", "V07020102", 
    "V07020501", "V07020601", "V07020801", "V07021101", "V07021201",
    "V07021301", "V07021501", "V07021601", "V09000001", "V09000002",
    "V09000003", "V09010101", "V09010102", "V09010103", "V09020201",
    "V09020202", "V09020203", "VW09000000", "VW09000001","VW09000002",
    "VW09000003", "V11030100", "V12030100", "V08010101", "V08010102",
    "V08010201", "V08010202", "V10010101", "V10010102", "V10010201",
    "V10010202", "V10020101", "V10020102", "V10020201", "V10020202",
    "V10030101", "V10030102", "V10030201", "V10030202", "VW10040101",
    "VW10040201", "VW08010304", "VW08010305", "VW08010306", "VW08010405", 
    "VW10010001", "VW10010002", "VW10010003", "VW10020001", "VW10020002",
    "VW10020003", "VW10030001", "VW10030002", "VW10030003", "VW10040001",
    "VW10040002", "VW10040003", "VW10040004", "VW10040005", "VW10040006",
    "VW10040007", "VW10000001", "V35000102", "V45010100", "V45010200",
    "V45010300", "V45010500", "V45010800", "V45010900", "V45011200",
    "V45011300", "V45011400", "V45011500", "V45011900", "V45012300",
    "V45014200", "V45012500", "V45012600", "V45012800", "VW45013600",
    "V46011400", "V46011500", "V46011600", "V46011800", "V46010300",
    "V46010700",
]

estabel_vars_numericas_mediana = ["V02200000", "V02200001"]

# Regras de imputação numérica para a base de estabelecimentos.
df_estabel[estabel_vars_numericas_zero] = df_estabel[estabel_vars_numericas_zero].fillna(0)
df_estabel[estabel_vars_numericas_mediana] = df_estabel[estabel_vars_numericas_mediana].fillna(
    df_estabel[estabel_vars_numericas_mediana].median())

In [ ]:
estabel_vars_categoricas_missing = [
    'V01170501',
    'V02010000',
    'V02020000',
    'V02090000',
    'V02120000',
    'V05030100',
    'V05150100',
    'V49010000',
]

# Campos categóricos com ausência equivalente a resposta "não".
estabel_vars_categoricas_nao = [
    'V01170500',
    'V05010100',
    'V02100000',
    'V05020100',
    'V05100100',
    'V05130100',
    'V05130300',
    'V05130500',
    'V05130700',
    'V05130900',
    'V05131000',
    'V05131100',
    'V05131700',
    'V05131900',
    'V05132000',
    'V05140100',
    'V05180100',
    'V05270100',
    'V05280100',
    'V05290100',
    'V05310100',
    'V05330100',
    'V05390100',
    'V06010100',
    'V07010100',
    'V07020100',
    'V07021100',
    'V12010000',
    'V10010100',
    'V10020100',
    'V10030100',
    'V13010000',
    'V13011400',
    'V13011500',
    'V13011600',
    'V13011700',
    'V13011800',
    'V13011900',
    'V13012000',
    'V13012100',
    'V13012200',
    'V13012500',
    'V13012400',
    'V13012700',
    'V13012800',
    'V13012900',
    'V13013000',
    'V13013100',
    'V13013200',
    'V33010000',
    'V33010100',
    'V33010200',
    'V33010300',
    'V33010400',
    'V33010500',
    'V33010600',
    'V33010700',
    'V43010100',
    'V45010000',
    'V46010000',
    'V46010001',
]

estabel_vars_categoricas_vazias = [
    'V02240000',
    'V02210000',
    'V02220000',
    'V02230000',
    'V02210001',
    'V02220001',
    'V02230001',
    'V02070000',
    'V02120100',
    'V02120200',
    'V04020001',
    'V04050001',
    'V04080001',
    'V02180100',
    'V02180200',
    'V02180800',
    'V02180300',
    'V02181000',
    'V02180500',
    'V02180400',
    'V02180700',
    'V02180600',
    'V02181200',
    'V02181100',
    'V05020300',
    'V05020400',
    'V05020500',
    'V05020600',
    'V05120100',
    'V05120200',
    'V05120300',
    'V05120400',
    'V05120500',
    'V05120600',
    'V05120700',
    'V05120800',
    'V05400100',
    'V05140200',
    'V05400300',
    'V05400400',
    'V05400500',
    'V05400600',
    'V05400700',
    'V05400800',
    'V05160100',
    'V05250100',
    'V05270101',
    'V05270102',
    'V05270103',
    'V05310102',
    'V05310103',
    'V05310104',
    'V05310105',
    'V05340001',
    'V05340002',
    'V05340003',
    'V05340004',
    'V09010100',
    'V09010200',
    'V12010101',
    'V12010102',
    'V12010103',
    'V12010104',
    'V13060000',
    'V13080000',
    'V13080101',
    'V13080102',
    'V13080103',
    'V35000100',
    'V41010100',
    'V41020100',
    'V43020100',
    'V43020200',
    'V43020300',
    'V43020400',
    'V43030000',
    'V43030101',
    'V43030105',
    'V43030106',
    'V43030107',
    'V43030108',
    'V43030110',
    'V43030109',
    'V43040100',
    'V43040200',
    'V43040300',
    'V43040400',
    'V43040500',
    'V43040600',
    'V43040700',
    'V43040800',
    'V43040900',
    'V43041000',
    'V46010601',
    'V51020100',
]

# missing + vazias: preencher nulos e converter para categórica
for col in estabel_vars_categoricas_missing + estabel_vars_categoricas_vazias:
    df_estabel[col] = df_estabel[col].astype('string').fillna(VALOR_CATEGORICO_VAZIO).astype('category')

# naos: preencher nulos com valor de ausência equivalente a "não"
for col in estabel_vars_categoricas_nao:
    df_estabel[col] = df_estabel[col].astype('string').fillna(VALOR_CATEGORICO_NAO).astype('category')

Adicionando a lógica de cálculo da % das áreas em relação a área total do estabelecimento das regras 23 e 24

In [ ]:
# Criando a coluna de area de pastagem boa em ha
df_estabel['area_pastagem_boa_ha'] = df_estabel['VW04060000'] + df_estabel['VW04050000']

var_area_estabel_ha = "VW01170300"

# Chave = código da coluna | Valor = descrição (na mesma ordem do notebook)
cols_para_calcular_perc_area = {
    "VW04020000": "perc_area_lavoura_permanente_ha",
    "VW04030000": "perc_area_lavoura_temporaria_ha",
    "VW04040000": "perc_area_cultivo_flores_viveiros_estufas_casas_vegetacao_ha",
    "VW04050000": "perc_area_pastagem_natural_ha",
    "VW04060000": "perc_area_pastagem_plantada_ha",
    "VW04070000": "perc_area_pastagem_degradada_ou_ma_condicao_ha",
    "VW04080000": "perc_area_matas_florestas_naturais_preservacao_permanente_reserva_legal_ha",
    "VW04090000": "perc_area_matas_florestas_naturais_ha",
    "VW04100000": "perc_area_florestas_plantadas_ha",
    "VW04110000": "perc_area_especies_florestais_integracao_lavoura_floresta_pecuaria_ha",
    "VW04120000": "perc_area_lamina_agua_aquicultura_construcoes_benfeitorias_caminhos_terras_degradadas_inaproveitaveis_ha",
    "VW04160000": "perc_area_lavouras_ha",
    "VW04170000": "perc_area_pastagens_ha",
    "VW04180000": "perc_area_matas_ha",
    "VW05270104": "perc_area_plantio_direto_ha",
    "area_pastagem_boa_ha": "perc_area_pastagem_boa_ha"
}
denominador = df_estabel[var_area_estabel_ha].replace(0, pd.NA)

df_perc_areas = (
    df_estabel[list(cols_para_calcular_perc_area.keys())]
    .div(denominador, axis=0)
    .rename(columns=cols_para_calcular_perc_area)
    .fillna(0)
)

df_estabel = pd.concat([df_estabel, df_perc_areas], axis=1)

df_estabel

Criando a flag para melhorar a detecção da regra 25

In [ ]:
# Query para identificar estabelecimentos que fazem plantio direto orgânico
chaves_plantio_direto_organico = set(
    map(tuple, df_estabel[
        (df_estabel['V05270103'] == '2')
        & (df_estabel['VW05270104'] > 0)
        & (df_estabel['V05250100'].isin(['2', '4']))
    ][COLUNHAS_CHAVES].to_numpy())
)

# Criar a flag no dataframe principal
df_estabel['faz_plantio_direto_organico'] = (
    df_estabel[COLUNHAS_CHAVES]
    .apply(tuple, axis=1)
    .isin(chaves_plantio_direto_organico)
    .astype('int8')
    .astype('str')
    .astype('category')
)

df_estabel['faz_plantio_direto_organico'].value_counts()


# Lavoura Temporária

In [ ]:
query = """
SELECT
    *
FROM lav_temp
"""

lav_temp_cols_descartar = [
    'V34020300',
    'V34020400',
    'V34020401',
    'V34020500',
    'V34020600',
    'V34020700',
    'V34020800',
    'VW34020301',
    'VW34020100',
]

with sqlite3.connect('../data/amstr_geral_criticas.db') as conn:
    df_lav_temp = pd.read_sql_query(query, conn)

# # Mantém apenas colunas úteis e remove eventuais duplicidades de schema.
# df_lav_temp = df_lav_temp.loc[:, ~df_lav_temp.columns.isin(lav_temp_cols_descartar)]
# df_lav_temp = df_lav_temp.loc[:, ~df_lav_temp.columns.duplicated()]

# # Merge com a base mestre de estabelecimentos.
# df_lav_temp = pd.merge(df_estabel, df_lav_temp, on=COLUNHAS_CHAVES, how='left')

# # Garante ausência de nulos imediatamente após merge.
# df_lav_temp = preencher_nulos_dataframe(df_lav_temp)

df_lav_temp

In [ ]:
lav_temp_vars_numericas_zero = [
    'V34020900', 'VW04020000', 'VW34020900', 'VW34020901', 
    'VW34020902', 'VW34020500', 'VW34020600', 'VW34020300',
    'VW34020501'
]

df_lav_temp[lav_temp_vars_numericas_zero] = df_lav_temp[lav_temp_vars_numericas_zero].fillna(0)

In [ ]:
lav_temp_vars_categoricas_missing = []
lav_temp_vars_categoricas_vazias = ['V34021700']

# missing + vazias: preencher nulos e converter para categórica
for col in lav_temp_vars_categoricas_missing + lav_temp_vars_categoricas_vazias:
    df_lav_temp[col] = df_lav_temp[col].astype('string').fillna(VALOR_CATEGORICO_VAZIO).astype('category')

Adicionando as seguintes variáveis para o processamento:
- Diferença da área total e área colhida
- Razão entre a diferença da área total e área colhida

In [ ]:
# Agrega por estabelecimento para obter área total (primeiro valor) e área colhida total (soma).
df_lav_temp_agregado = (
    df_lav_temp
    .groupby(COLUNHAS_CHAVES, as_index=False)
    .agg(
        VW04020000=('VW04020000', 'first'),
        total_colhida=('VW34020300', 'sum')
    )
)

# Remove a área total do agregado para evitar duplicidade após o merge com df_lav_temp.
df_lav_temp_agregado.drop('VW04020000', axis=1, inplace=True)
df_lav_temp_agregado = df_lav_temp_agregado.fillna({'total_colhida': 0})

display(df_lav_temp_agregado)

# Anexa agregados por estabelecimento e saneia nulos após merge.
df_lav_temp = df_lav_temp.merge(df_lav_temp_agregado, on=COLUNHAS_CHAVES, how='left')
df_lav_temp = preencher_nulos_dataframe(df_lav_temp)

In [ ]:
#TODO: se necessário criar a flag do plantio sem área declarada

In [ ]:
# TALVEZ DEPOIS SERIA INTERESSANTE APLICAR DIRETAMENTE O RECORTE ESPERADO GERANDO UMA VARIÁVEL BOOL

# Calcula a diferenca absoluta entre area total declarada e area total colhida.
df_lav_temp['diff_area_total_area_colhida'] = (
    df_lav_temp['VW04020000'] - df_lav_temp['total_colhida']
).round(2)

# Calcula a razao apenas para diferencas negativas; demais ficam com 0.
mask_neg = df_lav_temp['diff_area_total_area_colhida'] < 0
df_lav_temp['razao_diff_area_total_area_colhida'] = 0.0
df_lav_temp.loc[mask_neg, 'razao_diff_area_total_area_colhida'] = (
    df_lav_temp.loc[mask_neg, 'diff_area_total_area_colhida'].abs()
    / df_lav_temp.loc[mask_neg, 'VW04020000'].replace(0, pd.NA)
).fillna(0)

df_lav_temp['razao_diff_area_total_area_colhida'].describe()

Adicionando as flags que verificam:
- Estabelecimentos que declararam realizar suplementação alimentar sem despesas com a mesma e, não tem produção de culturas forrageiras.
- Estabelecimentos que declararam possuir culturas forrageiras e não apresentam criação de animais

In [ ]:
# Flag: suplementação alimentar sem despesa e sem culturas próprias de alimentação.
flag_col = 'sem_criacao_propria_alimentacao'
codigos_culturas_alvo = {262, 263, 264, 266}

condicao_base = (
    df_lav_temp['V13080102'].astype('string').eq('2')
    & pd.to_numeric(df_lav_temp['V45011500'], errors='coerce').fillna(0).eq(0)
)

df_base = df_lav_temp.loc[condicao_base, COLUNHAS_CHAVES + ['V34010100']].copy()

# Normaliza códigos de cultura para uma linha por código e por estabelecimento.
df_codes = (
    df_base.assign(V34010100=df_base['V34010100'].astype('string').fillna(''))
    .assign(V34010100=lambda d: d['V34010100'].str.split(','))
    .explode('V34010100')
)

df_codes['V34010100'] = pd.to_numeric(
    df_codes['V34010100'].astype('string').str.strip(),
    errors='coerce'
).astype('Int64')

df_intersecao = (
    df_codes.assign(eh_cultura_alvo=df_codes['V34010100'].isin(codigos_culturas_alvo))
    .groupby(COLUNHAS_CHAVES, as_index=False)['eh_cultura_alvo']
    .any()
)

chaves_sem_intersecao = set(
    map(
        tuple,
        df_intersecao.loc[~df_intersecao['eh_cultura_alvo'], COLUNHAS_CHAVES].to_numpy()
    )
)

df_lav_temp[flag_col] = (
    df_lav_temp[COLUNHAS_CHAVES]
    .apply(tuple, axis=1)
    .isin(chaves_sem_intersecao)
    .astype('int8')
    .astype('str')
    .astype('category')
)

df_lav_temp[flag_col].value_counts()

In [ ]:
df_lav_temp

# Pecuária

In [ ]:
query = """
SELECT * FROM pecu
"""

pecu_vars_to_ignore = [ "V29020500" ]

pecu_vars_nulas = [
    "VW14010103", "VW15010100", "VW16010100", "VW17101000",
    "VW18101000", "VW19010000", "VW19010900", "VW20010000",
    "V21050401",  "VW21010000", "VW21050201", "VW21050301",
    "VW22010101", "VW22050101", "VW22080101", "VW25010101",
    "V29070100",  "V29070201",  "V29070202",  "V29080100",
    "V29080201",  "V29080202",  "VW29070201",  "VW29080201",
    "V30010100",  "V30010201",  "V30010202",  "VW30010200",  
    "VW14130001",  "VW14130002",  "VW14130003",  "VW22080401",
]

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    df_pecu = pd.read_sql_query(query, conn).drop(columns=["id"])

df_pecu = df_pecu.loc[:, ~df_pecu.columns.isin(pecu_vars_to_ignore + pecu_vars_nulas)]
df_pecu = df_pecu.loc[:, ~df_pecu.columns.duplicated()]

df_pecu

In [ ]:
# Integra variáveis do estabelecimento para enriquecer as regras da pecuária.
df_pecu = pd.merge(df_pecu, df_estabel, on=COLUNHAS_CHAVES, how='left')

# Saneia nulos gerados no merge.
df_pecu = preencher_nulos_dataframe(df_pecu)
df_pecu

In [ ]:
pecu_vars_numericas_zero = [
    "V14010101", "V14090302", "V14130101", "V14130102", "V14130201", "V14130202",
    "V14130301", "V14130302", "V14130401", "V14130402", "VW14130000", "VW14130100",
    "VW14130200", "VW14130300", "VW14130400", "V14160101", "V14160201", "V14160301",
    "V14160401", "VW14160200", "VW14160300", "VW14160000", "VW14060000", "V15010101",
    "V15010501", "V15010502", "V15020101", "V15020201", "V15020301", "V15020401",
    "VW15020200", "VW15020300", "VW15020100", "VW15010500", "VW15010501", "V16010101",
    "V16010501", "V16010502", "VW16010500", "VW16010501", "V17010101", "V17010501",
    "V17010502", "VW17010500", "VW17010501", "V18010101", "V18010501", "V18010502",
    "VW18010500", "VW18010501", "V19010101", "V19020101", "V19020201", "V19020301",
    "V19030401", "V19030402", "VW19030400", "VW19030401", "V20010101", "V20030401",
    "V20030402", "VW20030400", "V20040101", "V20040201", "V20040301", "V20040401",
    "VW20040200", "VW20040201", "VW20040301", "V21010101", "V21030401", "V21030402",
    "VW21030400", "V21040101", "V21040201", "V21040301", "V21040401", "V21050101",
    "V21050201", "V21050301", "VW21040200", "VW21040201", "VW21040301", "VW21050200",
    "V22010101", "V22020101", "V22020201", "V22020301", "V22020401", "V22050001",
    "V22050002", "V22080101", "V22080401", "V22080402", "VW22050300", "VW22080100",
    "VW22080400", "V25010101", "V25030101", "V25030102", "V25040101", "V25040201",
    "V25040202", "VW25030000", "VW25040100", "VW25040200", "V23010101", "V24010101",
    "V26010101", "V26030101", "V26030102", "V26040201", "V26040202", "VW26030101",
    "VW26040201", "V27010101", "V27010301", "V27010302", "VW27010300", "V28020201",
    "V28020202", "V28030201", "V28030202", "V28040101", "V28080101", "VW28020200",
    "VW28030200", "VW28080101", "V29041602", "V29050201", "V29050202", "V29060201",
    "V29060202", "VW29050201", "VW29060201", "V31010101", "V31010201", "V31010202",
    "VW31010100", "VW31010200", "V32040100", "VW29020500"
]

df_pecu[pecu_vars_numericas_zero] = df_pecu[pecu_vars_numericas_zero].fillna(0)

In [ ]:
pecu_vars_categoricas_missing = ['V14020101']

pecu_vars_categoricas_vazias = [
    'V14010000', 'V14040000', 'V14040101', 'V14040103', 'V14130000',
    'V14160000', 'V15010000', 'V15010102', 'V15020000', 'V16010000',
    'V16010102', 'V17010000', 'V17010102', 'V18010000', 'V18010102',
    'V19010000', 'V19010102', 'V20010000', 'V20010102', 'V20040000',
    'V21010000', 'V21010102', 'V21040000', 'V21050000', 'V22010000',
    'V22050000', 'V22080000', 'V25010000', 'V25030000', 'V25040000',
    'V26010000', 'V26030000', 'V26040000', 'V27010000', 'V27010300',
    'V28020000', 'V28030000', 'V28040000', 'V28080000', 'V29020100',
    'V29020200', 'V29020300', 'V29020400', 'V29020600', 'V29040100',
    'V29040200', 'V29040300', 'V29040400', 'V29040500', 'V29040600',
    'V29040700', 'V29042100', 'V29040800', 'V29040900', 'V29041000',
    'V29041100', 'V29041200', 'V29041300', 'V29041400', 'V29041500',
    'V29041600', 'V29041700', 'V29041800', 'V29041900', 'V29042000',
    'V29050100', 'V29060100', 'V31010000', 'V32020100', 'V32030100', 
    'V32040000'
]

# missing + vazias: preencher nulos e converter para categórica
for col in pecu_vars_categoricas_missing + pecu_vars_categoricas_vazias:
    df_pecu[col] = df_pecu[col].astype('string').fillna(VALOR_CATEGORICO_VAZIO).astype('category')

Criando as flags para melhorar a detecção da regra 12

In [ ]:
df_pecu['vacas_reprodutoras_sem_producao_leite'] = (df_pecu['V14090302'] > 872).astype('int8').astype('str').astype('category')

df_pecu.value_counts('vacas_reprodutoras_sem_producao_leite')

In [ ]:
denominador_bovinos = pd.to_numeric(df_pecu['V14010101'], errors='coerce').replace(0, pd.NA)

df_pecu['relacao_area_qtd_bovinos'] = (
    pd.to_numeric(df_pecu['VW01170300'], errors='coerce')
    .div(denominador_bovinos)
    .fillna(0)
)

df_pecu['relacao_area_lav_qtd_bovinos'] = (
    pd.to_numeric(df_pecu['VW04160000'], errors='coerce')
    .div(denominador_bovinos)
    .fillna(0)
)

df_pecu['relacao_area_qtd_bovinos_baixa'] = (df_pecu['relacao_area_qtd_bovinos'] < 0.51).astype('int8').astype('str').astype('category')
df_pecu['relacao_area_lav_qtd_bovinos_baixa'] = (df_pecu['relacao_area_lav_qtd_bovinos'] < 0.0018).astype('int8').astype('str').astype('category')

display(df_pecu['relacao_area_qtd_bovinos_baixa'].value_counts())
display(df_pecu['relacao_area_lav_qtd_bovinos_baixa'].value_counts())

Criando a flag para melhorar a detecção da regra 13

In [ ]:
cols_condicao_estabel = [
    'V13011400', 'V13011500', 'V13011600', 'V13011700',
    'V13011800', 'V13011900', 'V13012000', 'V13012100',
    'V13012200', 'V13012400', 'V13012500', 'V13012700',
    'V13012800', 'V13012900', 'V13013000', 'V13013100'
]

# Mantém apenas colunas de condição existentes no dataframe.
cols_condicao_estabel_existentes = [c for c in cols_condicao_estabel if c in df_pecu.columns]

# Condição equivalente a p.V22020401 > 0.
mask_pecu = pd.to_numeric(df_pecu['V22020401'], errors='coerce').fillna(0).gt(0)

# Condição equivalente ao bloco OR de variáveis V13011xxxx = '2'.
if cols_condicao_estabel_existentes:
    mask_estabel = df_pecu[cols_condicao_estabel_existentes].astype('string').eq('2').any(axis=1)
else:
    mask_estabel = pd.Series(False, index=df_pecu.index)

chaves_flag = set(
    map(
        tuple,
        df_pecu.loc[mask_pecu & mask_estabel, COLUNHAS_CHAVES]
        .drop_duplicates()
        .to_numpy()
    )
)

df_pecu['plantel_galinhas_matrizes_descomunal'] = (
    df_pecu[COLUNHAS_CHAVES].apply(tuple, axis=1)
    .isin(chaves_flag)
    .astype('int8')
    .astype('str')
    .astype('category')
)

df_pecu['plantel_galinhas_matrizes_descomunal'].value_counts()

Criando a flag para melhorar a detecção da regra 15

In [ ]:
# Regra 15: produção animal declarada sem efetivo correspondente.
colunas_regra_15 = [
    'V19010000', 'V19010101', 'V22010000', 'V22010101', 'V14010000', 'V14010101',
    'V15010000', 'V15010101', 'V16010000', 'V16010101', 'V17010000', 'V17010101',
    'V18010000', 'V18010101', 'V20010000', 'V20010101', 'V21010000', 'V21010101',
    'V25010000', 'V25010101', 'V27010000', 'V27010101', 'V26010000', 'V26010101',
    'V24010101', 'V23010101', 'V28080000', 'V28080101', 'V13011400', 'V13011500',
    'V13011600', 'V13011700', 'V13011800', 'V13011900', 'V13012000', 'V13012100',
    'V13012200', 'V13012400', 'V13012500', 'V13012700', 'V13012800', 'V13013100',
    'V31010101'
]

for col in colunas_regra_15:
    if col not in df_pecu.columns:
        df_pecu[col] = pd.NA

cond_pecu = (
    (df_pecu['V19010000'].astype('string').eq('2') & pd.to_numeric(df_pecu['V19010101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V22010000'].astype('string').eq('2') & pd.to_numeric(df_pecu['V22010101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V14010000'].astype('string').eq('2') & pd.to_numeric(df_pecu['V14010101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V15010000'].astype('string').eq('2') & pd.to_numeric(df_pecu['V15010101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V16010000'].astype('string').eq('2') & pd.to_numeric(df_pecu['V16010101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V17010000'].astype('string').eq('2') & pd.to_numeric(df_pecu['V17010101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V18010000'].astype('string').eq('2') & pd.to_numeric(df_pecu['V18010101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V20010000'].astype('string').eq('2') & pd.to_numeric(df_pecu['V20010101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V21010000'].astype('string').eq('2') & pd.to_numeric(df_pecu['V21010101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V25010000'].astype('string').eq('2') & pd.to_numeric(df_pecu['V25010101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V27010000'].astype('string').eq('2') & pd.to_numeric(df_pecu['V27010101'], errors='coerce').fillna(0).eq(0)) |
    (
        df_pecu['V26010000'].astype('string').eq('2')
        & pd.to_numeric(df_pecu['V26010101'], errors='coerce').fillna(0).eq(0)
        & pd.to_numeric(df_pecu['V24010101'], errors='coerce').fillna(0).eq(0)
        & pd.to_numeric(df_pecu['V23010101'], errors='coerce').fillna(0).eq(0)
    ) |
    (df_pecu['V28080000'].astype('string').eq('2') & pd.to_numeric(df_pecu['V28080101'], errors='coerce').fillna(0).eq(0))
)

cond_estabel = (
    (df_pecu['V13011900'].astype('string').eq('2') & pd.to_numeric(df_pecu['V19010101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V13012200'].astype('string').eq('2') & pd.to_numeric(df_pecu['V22010101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V13011400'].astype('string').eq('2') & pd.to_numeric(df_pecu['V14010101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V13011500'].astype('string').eq('2') & pd.to_numeric(df_pecu['V15010101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V13011600'].astype('string').eq('2') & pd.to_numeric(df_pecu['V16010101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V13011700'].astype('string').eq('2') & pd.to_numeric(df_pecu['V17010101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V13011800'].astype('string').eq('2') & pd.to_numeric(df_pecu['V18010101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V13012000'].astype('string').eq('2') & pd.to_numeric(df_pecu['V20010101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V13012100'].astype('string').eq('2') & pd.to_numeric(df_pecu['V21010101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V13012500'].astype('string').eq('2') & pd.to_numeric(df_pecu['V25010101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V13012700'].astype('string').eq('2') & pd.to_numeric(df_pecu['V27010101'], errors='coerce').fillna(0).eq(0)) |
    (
        df_pecu['V13012400'].astype('string').eq('2')
        & pd.to_numeric(df_pecu['V26010101'], errors='coerce').fillna(0).eq(0)
        & pd.to_numeric(df_pecu['V24010101'], errors='coerce').fillna(0).eq(0)
        & pd.to_numeric(df_pecu['V23010101'], errors='coerce').fillna(0).eq(0)
    ) |
    (df_pecu['V13012800'].astype('string').eq('2') & pd.to_numeric(df_pecu['V28080101'], errors='coerce').fillna(0).eq(0)) |
    (df_pecu['V13013100'].astype('string').eq('2') & pd.to_numeric(df_pecu['V31010101'], errors='coerce').fillna(0).eq(0))
)

chaves_producao_animal_sem_efetivo = set(
    map(
        tuple,
        pd.concat(
            [
                df_pecu.loc[cond_pecu, COLUNHAS_CHAVES],
                df_pecu.loc[cond_estabel, COLUNHAS_CHAVES]
            ],
            ignore_index=True
        ).drop_duplicates().to_numpy()
    )
)

df_pecu['producao_animal_sem_efetivo'] = (
    df_pecu[COLUNHAS_CHAVES].apply(tuple, axis=1)
    .isin(chaves_producao_animal_sem_efetivo)
    .astype('int8')
    .astype('str')
    .astype('category')
)

df_pecu['producao_animal_sem_efetivo'].value_counts()

Criando a flag para melhorar a detecção da regra 17

In [ ]:
colunas_ruminantes = [
    'V14010000', 'V15010000', 'V20010000', 'V21010000',
    'V18010000', 'V16010000', 'V17010000'
]

for col in colunas_ruminantes:
    if col not in df_pecu.columns:
        df_pecu[col] = pd.NA

cond_ruminante = (
    (df_pecu['V14010000'].astype('string').eq('2') | pd.to_numeric(df_pecu['V14010000'], errors='coerce').fillna(0).eq(0))
    & (df_pecu['V15010000'].astype('string').eq('2') | pd.to_numeric(df_pecu['V15010000'], errors='coerce').fillna(0).eq(0))
    & (df_pecu['V20010000'].astype('string').eq('2') | pd.to_numeric(df_pecu['V20010000'], errors='coerce').fillna(0).eq(0))
    & (df_pecu['V21010000'].astype('string').eq('2') | pd.to_numeric(df_pecu['V21010000'], errors='coerce').fillna(0).eq(0))
    & (df_pecu['V18010000'].astype('string').eq('2') | pd.to_numeric(df_pecu['V18010000'], errors='coerce').fillna(0).eq(0))
    & (df_pecu['V16010000'].astype('string').eq('2') | pd.to_numeric(df_pecu['V16010000'], errors='coerce').fillna(0).eq(0))
    & (df_pecu['V17010000'].astype('string').eq('2') | pd.to_numeric(df_pecu['V17010000'], errors='coerce').fillna(0).eq(0))
)

cond_area_pastagem_boa = df_pecu['perc_area_pastagem_boa_ha'] > 0.8

df_pecu['area_pastagem_boa_mas_sem_efetivo_ruminantes'] = (
    cond_area_pastagem_boa & cond_ruminante
).astype('int8').astype('str').astype('category')

df_pecu['area_pastagem_boa_mas_sem_efetivo_ruminantes'].value_counts()

Criando a flag para melhorar a detecção da regra 18

In [ ]:
df_pecu['area_pastagem_natural_extensa_sem_efetivo_ruminantes'] = (
    (df_pecu['perc_area_pastagem_natural_ha'] > 0.9) &
    cond_ruminante
).astype('int8').astype('str').astype('category')

Criando a flag para melhorar a detecção da regra 31

In [ ]:
# Regra 31: aração/gradagem sem maquinário ou animais compatíveis.
colunas_regra_31 = [
    'V05270101', 'V07020100', 'V13011600', 'V13011700',
    'V13011800', 'V13011500', 'V13011400', 'V14020101'
]

for col in colunas_regra_31:
    if col not in df_pecu.columns:
        df_pecu[col] = pd.NA

v14020101 = pd.to_numeric(df_pecu['V14020101'], errors='coerce')
mask_aracao = (
    df_pecu['V05270101'].astype('string').eq('2')
    & df_pecu['V07020100'].astype('string').eq('1')
    & df_pecu['V13011600'].astype('string').eq('1')
    & df_pecu['V13011700'].astype('string').eq('1')
    & df_pecu['V13011800'].astype('string').eq('1')
    & df_pecu['V13011500'].astype('string').eq('1')
    & (
        df_pecu['V13011400'].astype('string').eq('1')
        | (v14020101.notna() & v14020101.ne(6))
    )
)

chaves_aracao = set(
    map(tuple, df_pecu.loc[mask_aracao, COLUNHAS_CHAVES].drop_duplicates().to_numpy())
)

df_pecu['aracao_gradagem_sem_maquinario_ou_animais'] = (
    df_pecu[COLUNHAS_CHAVES]
    .apply(tuple, axis=1)
    .isin(chaves_aracao)
    .astype('int8')
    .astype('str')
    .astype('category')
)

df_pecu['aracao_gradagem_sem_maquinario_ou_animais'].value_counts()

## Validação final de consistência

> Garantia explícita de ausência de nulos nos dataframes finais gerados neste notebook.

In [ ]:
# Aplica saneamento final para garantir saída sem nulos.
df_estabel = preencher_nulos_dataframe(df_estabel)
df_lav_temp = preencher_nulos_dataframe(df_lav_temp)
df_pecu = preencher_nulos_dataframe(df_pecu)

resumo_nulos = pd.Series(
    {
        'df_estabel_nulos': int(df_estabel.isna().sum().sum()),
        'df_lav_temp_nulos': int(df_lav_temp.isna().sum().sum()),
        'df_pecu_nulos': int(df_pecu.isna().sum().sum()),
    }
)

display(resumo_nulos)
assert resumo_nulos.sum() == 0, 'Ainda existem valores nulos nos dataframes finais.'

In [ ]:
print(df_estabel.info())
print()
print(df_lav_temp.info())
print()
print(df_pecu.info())

In [ ]:
import json

def list_int64_columns(df):    
    mapa_variavel = json.load(open('../data/mapa_variaveis.json', 'r'))

    # Selecionando todas as colunas int64
    colunas_int64 = df.select_dtypes(include=['int64']).columns
    for col in colunas_int64:
        if col in COLUNHAS_CHAVES:
            continue

        print(f"Coluna: {col} - {mapa_variavel.get(col, {}).get('name', 'Sem descrição')}")
        # print(f"    - Valores únicos: {df[col].unique()}")

list_int64_columns(df_estabel)
print()
list_int64_columns(df_lav_temp)
print()
list_int64_columns(df_pecu)

# Exportando os Dataframes para Parquet

In [ ]:
df_estabel.to_parquet('../data/experimentos/abordagem1/df_estabel_final.parquet', index=False)
df_lav_temp.to_parquet('../data/experimentos/abordagem1/df_lav_temp_final.parquet', index=False)
df_pecu.to_parquet('../data/experimentos/abordagem1/df_pecu_final.parquet', index=False)